In [1]:
import cv2
import numpy as np
from PIL import Image
from skimage.filters import threshold_multiotsu
from skimage.segmentation import active_contour
from skimage.filters import gaussian
from skimage import img_as_float


cale_imagine = r'/Users/ana/Documents/Lung Cancer Diagnosis System/Imagini pentru analiza/cancere/Tumora cu invazie perete toracic.bmp'

TUMOR_IS_DARK = True

# contur alb
ROI_CLAHE_CLIP = 2.0
ROI_GAUSS = (9, 9)
ROI_MORPH_KERNEL = (10, 10)

# Preprocesare tumori
TUM_CLAHE_CLIP = 3.0
TUM_MEDIAN_K = 5
MULTI_OTSU_CLASSES = 3

# Curățare mască tumoare
OPEN_K = (7, 7)
CLOSE_K = (11, 11)
OPEN_IT = 1
CLOSE_IT = 2
ROI_ERODE_BORDER = 25
MIN_TUMOR_AREA = 350

# Snake 
RUN_SNAKE = True
SNAKE_ALPHA = 0.01
SNAKE_BETA  = 0.01
SNAKE_GAMMA = 0.001
SNAKE_W_EDGE = 1000.0
SNAKE_W_LINE = -100.0

RUN_WHITE_LINES = True
#prinde structuri luminoase mai lungi
TOPHAT_K = (31, 31) 

WHITE_PERC = 90     
WHITE_MIN_AREA = 90 

def largest_component(mask255, min_area=200):
    """Păstrează cea mai mare componentă conexă din mască (0/255)."""
    mask = (mask255 > 0).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return np.zeros_like(mask255)
    areas = stats[1:, cv2.CC_STAT_AREA]
    best = 1 + int(np.argmax(areas))
    if stats[best, cv2.CC_STAT_AREA] < min_area:
        return np.zeros_like(mask255)
    return (labels == best).astype(np.uint8) * 255


def fill_holes(mask255):
    """Umple găurile din mască (0/255)"""
    m = mask255.copy()
    h, w = m.shape
    flood = m.copy()
    ffmask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, ffmask, (0, 0), 255)
    flood_inv = cv2.bitwise_not(flood)
    return cv2.bitwise_or(m, flood_inv)


def run_snake(gray, init_mask255):
    """Snake inițializat din conturul măștii."""
    cnts, _ = cv2.findContours(init_mask255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None

    c = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(c) < 300:
        return None

    init = c.squeeze().astype(float)  # (N,2) x,y
    if init.ndim != 2 or init.shape[0] < 30:
        return None

    init_rc = np.fliplr(init)  # (y,x)
    img = gaussian(img_as_float(gray), sigma=1.0, preserve_range=False)

    snake = active_contour(
        img, init_rc,
        alpha=SNAKE_ALPHA, beta=SNAKE_BETA, gamma=SNAKE_GAMMA,
        w_edge=SNAKE_W_EDGE, w_line=SNAKE_W_LINE
    )
    return snake


def detect_white_lines(gray, roi_mask, tumor_mask=None):
    """
    Detectează structuri BRIGHT (linii albe) inclusiv CURBE.
    """
    g = gray.copy()

    # Top-hat: scoate în evidență structuri luminoase pe fundal gri
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, TOPHAT_K)
    tophat = cv2.morphologyEx(g, cv2.MORPH_TOPHAT, k)
    tophat = cv2.normalize(tophat, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # doar ROI
    roi_vals = tophat[roi_mask == 255]
    if roi_vals.size < 200:
        return np.zeros_like(gray, np.uint8), []

    # Prag dinamic: prinde ce e "cel mai bright" 
    thr = np.percentile(roi_vals, WHITE_PERC)

    binm = (tophat >= thr).astype(np.uint8) * 255
    binm = cv2.bitwise_and(binm, binm, mask=roi_mask)

    # Curățare: unește bucăți și scoate zgomot
    binm = cv2.medianBlur(binm, 5)
    binm = cv2.morphologyEx(
        binm, cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)),
        iterations=1
    )
    binm = cv2.morphologyEx(
        binm, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)),
        iterations=1
    )

    # Păstrează doar componente suficient de mari
    n, labels, stats, _ = cv2.connectedComponentsWithStats((binm > 0).astype(np.uint8), 8)
    out = np.zeros_like(binm)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= WHITE_MIN_AREA:
            out[labels == i] = 255

    # Îngroașă puțin ca să se vadă bine în overlay
    out = cv2.dilate(out, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)), iterations=1)

    # Contururi (curbe)
    cnts, _ = cv2.findContours(out, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    smooth_cnts = []
    for c in cnts:
        if len(c) < 20:
            continue
        eps = 0.003 * cv2.arcLength(c, True)  # mic => păstrează curbura
        approx = cv2.approxPolyDP(c, eps, True)
        smooth_cnts.append(approx)

    return out, smooth_cnts


def overlay_safe(bgr, roi_mask, tumor_mask, roi_contour=None, tumor_contour=None, snake=None, lines_cnts=None):
    """Overlay fără erori addWeighted (folosește np.where) + contururi linii albe curbe."""
    overlay = bgr.copy()

    # roșu în ROI
    red_layer = np.zeros_like(bgr, dtype=np.uint8)
    red_layer[:, :, 2] = 255
    blend_roi = cv2.addWeighted(overlay, 0.75, red_layer, 0.25, 0)
    overlay = np.where(roi_mask[..., None] == 255, blend_roi, overlay)

    # verde pe tumoare
    if tumor_mask is not None:
        green_layer = np.zeros_like(bgr, dtype=np.uint8)
        green_layer[:, :, 1] = 255
        blend_t = cv2.addWeighted(overlay, 0.30, green_layer, 0.70, 0)
        overlay = np.where(tumor_mask[..., None] == 255, blend_t, overlay)

    # contur ROI alb
    if roi_contour is not None:
        cv2.drawContours(overlay, [roi_contour], -1, (255, 255, 255), 2)

    # contur tumoare galben
    if tumor_contour is not None:
        cv2.drawContours(overlay, [tumor_contour], -1, (0, 255, 255), 2)

    # contur snake roșu (dacă există)
    if snake is not None:
        snake_coords = np.array(snake).astype(np.int32)     # (y,x)
        snake_xy = np.fliplr(snake_coords)                  # (x,y)
        cv2.polylines(overlay, [snake_xy], True, (0, 0, 255), 2)

    # Linii albe detectate (curbe) - CYAN
    if lines_cnts is not None and len(lines_cnts) > 0:
        cv2.drawContours(overlay, lines_cnts, -1, (255, 255, 0), 2)

    return overlay


# MAIN
img_pil = Image.open(cale_imagine)
road_rgb = np.array(img_pil)
road_bgr = cv2.cvtColor(road_rgb, cv2.COLOR_RGB2BGR)
road_gray = cv2.cvtColor(road_bgr, cv2.COLOR_BGR2GRAY)


# 1) ROI (contur alb)
clahe_roi = cv2.createCLAHE(clipLimit=ROI_CLAHE_CLIP, tileGridSize=(8, 8))
img_contrast = clahe_roi.apply(road_gray)
img_blur_roi = cv2.GaussianBlur(img_contrast, ROI_GAUSS, 0)

_, th_roi = cv2.threshold(img_blur_roi, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
th_roi = cv2.morphologyEx(th_roi, cv2.MORPH_CLOSE, np.ones(ROI_MORPH_KERNEL, np.uint8))

cnts_roi, _ = cv2.findContours(th_roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
if not cnts_roi:
    raise SystemExit("Nu am găsit ROI (contur alb).")

roi_contour = max(cnts_roi, key=cv2.contourArea)
roi_mask = np.zeros_like(road_gray, dtype=np.uint8)
cv2.drawContours(roi_mask, [roi_contour], -1, 255, -1)
print("ROI (contur alb) calculat.")

# 2) Tumora AUTOMAT: Multi-Otsu + morfologie 
den = cv2.medianBlur(road_gray, TUM_MEDIAN_K)
clahe_t = cv2.createCLAHE(clipLimit=TUM_CLAHE_CLIP, tileGridSize=(8, 8))
con = clahe_t.apply(den)
con = cv2.bitwise_and(con, con, mask=roi_mask)

roi_vals = con[roi_mask == 255]
if roi_vals.size < 500:
    print("ROI prea mic / invalid pentru Multi-Otsu -> sar peste tumoare.")
    tumor_mask = np.zeros_like(road_gray, dtype=np.uint8)
else:
    thr = threshold_multiotsu(roi_vals, classes=MULTI_OTSU_CLASSES)
    regions = np.digitize(con, bins=thr)  # 0..classes-1

    tumor_class = 0 if TUMOR_IS_DARK else (MULTI_OTSU_CLASSES - 1)
    tumor_cand = (regions == tumor_class).astype(np.uint8) * 255
    tumor_cand = cv2.bitwise_and(tumor_cand, tumor_cand, mask=roi_mask)

    roi_eroded = cv2.erode(
        roi_mask,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ROI_ERODE_BORDER, ROI_ERODE_BORDER)),
        iterations=1
    )
    tumor_cand = cv2.bitwise_and(tumor_cand, tumor_cand, mask=roi_eroded)

    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, OPEN_K)
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, CLOSE_K)

    tumor_cand = cv2.morphologyEx(tumor_cand, cv2.MORPH_OPEN, k_open, iterations=OPEN_IT)
    tumor_cand = cv2.morphologyEx(tumor_cand, cv2.MORPH_CLOSE, k_close, iterations=CLOSE_IT)
    tumor_cand = fill_holes(tumor_cand)
    tumor_mask = largest_component(tumor_cand, min_area=MIN_TUMOR_AREA)

    if np.count_nonzero(tumor_mask) == 0:
        print("Nu am găsit o tumoare clară cu parametrii curenți.")
    else:
        print("Masca tumoare (multi-otsu) calculată.")

tumor_contour = None
cnts_t, _ = cv2.findContours(tumor_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
if cnts_t:
    tumor_contour = max(cnts_t, key=cv2.contourArea)

# 3) Snake 
snake = None
if RUN_SNAKE and np.count_nonzero(tumor_mask) > 0:
    try:
        snake = run_snake(road_gray, tumor_mask)
        if snake is not None:
            print("Snake rulat (finisaj contur).")
    except Exception as e:
        print("Snake error:", e)
        snake = None

# 4) WHITE LINES
lines_mask = np.zeros_like(road_gray, dtype=np.uint8)
lines_cnts = []
if RUN_WHITE_LINES:
    lines_mask, lines_cnts = detect_white_lines(road_gray, roi_mask, tumor_mask=tumor_mask)
    print(f"Zone/linii albe detectate (curbe): {len(lines_cnts)}")

# 5) Overlay
overlay = overlay_safe(
    road_bgr, roi_mask, tumor_mask,
    roi_contour=roi_contour,
    tumor_contour=tumor_contour,
    snake=snake,
    lines_cnts=lines_cnts
)

cv2.namedWindow("ROI mask", cv2.WINDOW_NORMAL)
cv2.namedWindow("Masca tumoare (finala)", cv2.WINDOW_NORMAL)
cv2.namedWindow("White lines mask", cv2.WINDOW_NORMAL)
cv2.namedWindow("Overlay final (cu linii curbe)", cv2.WINDOW_NORMAL)

cv2.imshow("ROI mask", roi_mask)
cv2.imshow("Masca tumora (finala)", tumor_mask)
cv2.imshow("White lines mask", lines_mask)
cv2.imshow("Overlay final (cu linii curbe)", overlay)

print("ESC = închide")
while True:
    if (cv2.waitKey(10) & 0xFF) == 27:
        break
cv2.destroyAllWindows()


ROI (contur alb) calculat.
Masca tumoare (multi-otsu) calculată.
Snake rulat (finisaj contur).
Zone/linii albe detectate (curbe): 59
ESC = închide


2026-02-12 22:18:49.961 Python[33706:4167926] +[IMKClient subclass]: chose IMKClient_Modern
2026-02-12 22:18:49.961 Python[33706:4167926] +[IMKInputSession subclass]: chose IMKInputSession_Modern


KeyboardInterrupt: 